# LC 76 — Minimum Window Substring

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Two pointers + frequency counters.
Expand right until all required characters are covered, then
shrink left as much as possible to get the minimum window
— track "formed" count to know when you have a valid window
without rechecking the entire map.
</div>

## Official Problem Statement

Given two strings `s` and `t` of lengths `m` and `n` respectively,
return the *minimum window substring* of `s` such that every
character in `t` (including duplicates) is included in the window.
If there is no such substring, return the empty string `""`.

**Constraints:**
- `m == s.length`
- `n == t.length`
- `1 <= m, n <= 10^5`
- `s` and `t` consist of uppercase and lowercase English letters
- The test cases are generated such that the answer is unique

## What This Is Actually Asking

Find the shortest contiguous chunk of s that contains every
character in t at least as many times as t requires. The sliding
window works because if a window is valid, making it larger stays
valid but can't improve the answer — so shrink from the left.
The `formed` counter avoids rescanning `have` vs `need` every step
by only incrementing when a character count exactly hits its
required level.

## Walk Through an Example by Hand

```
s = "ADOBECODEBANC"  t = "ABC"
need = {A:1, B:1, C:1}   required = 3

r=0 A: have={A:1}, A hits need -> formed=1
r=1 D: have={A:1,D:1}, no change to formed
r=2 O: have={...}, formed=1
r=3 B: have={B:1}, B hits need -> formed=2
r=4 E: no change
r=5 C: have={C:1}, C hits need -> formed=3  VALID!
  window "ADOBEC" len=6, best=(0,5)
  shrink: l=0 A, have[A]=1=need[A] so formed-- -> 2
  stop shrinking (formed<3)
r=6 O: formed=2
r=7 D: formed=2
r=8 E: formed=2
r=9 B: have[B]=2, formed stays 2 (need only 1)
r=10 A: have[A]=1 -> formed=3  VALID!
  window "DOBECODEBA" -> shrink
  l=1 D: not in need, shrink l=2
  l=2 O: not in need, shrink l=3
  l=3 B: have[B]=2>need[B]=1, shrink l=4
  l=4 E: not in need, shrink l=5
  l=5 C: have[C]=1=need[C] formed-- -> 2 stop
  best was (5,10) len=6, no improvement
r=11 N: no change
r=12 C: have[C]=1 -> formed=3  VALID!
  window (6,12) "ODEBANC" len=7 -> no
  shrink: l=6 O, l=7 D, l=8 E, l=9 B (have[B]=1)
  l=9 B -> have[B] becomes 1, no change (still>=need)
  l=10 A -> have[A]=0 < need -> formed-- stop
  best window (9,12) "BANC" len=4  NEW BEST

Answer: "BANC"
```

## The Picture

```
s:  A  D  O  B  E  C  O  D  E  B  A  N  C
    0  1  2  3  4  5  6  7  8  9  10 11 12
    L                 R              <- first valid window
                                        "ADOBEC" len=6

   shrink left (A no longer covered) ->

              L                    R  <- second valid
                                       "DOBECODEBA" -> shrink

                         L         R  <- "CODEBA" shrink more

                            L      R  <- "ODEBANC"

                               L   R  <- "BANC" len=4 BEST!

FORMED counter:
  0 -> 1 (A)  -> 2 (B)  -> 3 (C) [valid!]
  shrink -> 2  expand again -> 3 ... etc.
```

## When To Use This Pattern

- When asked for **minimum/maximum subarray/substring** satisfying
  a condition, think **sliding window**.
- When you need **all characters of t in s** (with counts), think
  **need/have frequency maps + formed counter**.
- When the window validity check is expensive, think **formed
  counter to avoid repeated map scans**.
- When the problem has "contains all", "covers", "includes"
  language, think **expand right / shrink left**.
- When constraints are 10^5, think **O(n) sliding window,
  not O(n^2) brute force**.

## The Approach

Build `need = Counter(t)` and `required = len(need)`. Slide right
pointer expanding the window, incrementing `formed` only when
`have[c] == need[c]` exactly. Once `formed == required`, record
the window if it is the smallest so far, then shrink from the
left — decrement `formed` when a character drops below its
required count — and repeat.

In [ ]:
from collections import defaultdict, Counter
from typing import Optional

In [ ]:
def test_harness(func):
    cases = [
        ("ADOBECODEBANC", "ABC", "BANC"),
        ("a", "a", "a"),
        ("a", "aa", ""),
        ("aa", "aa", "aa"),
        ("abc", "b", "b"),
        ("", "a", ""),
    ]
    passed = 0
    for i, (s, t, expected) in enumerate(cases):
        result = func(s, t)
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        else:
            print(
                f"  Case {i}: got '{result}', "
                f"expected '{expected}'"
            )
        print(f"  Case {i}: {status}")
    print(f"\nSummary: {passed}/{len(cases)} passed")

In [ ]:
def min_window(s: str, t: str) -> str:
    """
    Find minimum window substring of s containing all of t.

    Args:
        s: source string to search in
        t: target string whose chars must all be covered

    Returns:
        shortest substring of s covering all chars in t,
        or "" if impossible.

    Strategy:
        - need = Counter(t), required = len(need)
        - Expand right: if have[c] == need[c] -> formed++
        - While formed == required: record window, shrink left
    """
    if not s or not t:
        pass  # return ""

    need = Counter(t)
    required = len(need)
    have = defaultdict(int)
    formed = 0
    l = 0
    best = (float("inf"), 0, 0)  # (length, left, right)

    print(f"[DEBUG] need={dict(need)}, required={required}")

    for r, c in enumerate(s):
        have[c] += 1
        if c in need and have[c] == need[c]:
            formed += 1
            print(
                f"[DEBUG] r={r} c={c}: formed -> {formed}"
            )

        while formed == required:
            window_len = r - l + 1
            if window_len < best[0]:
                best = (window_len, l, r)
                print(
                    f"[DEBUG] new best window "
                    f"'{s[l:r+1]}' len={window_len}"
                )
            # shrink left
            left_c = s[l]
            have[left_c] -= 1
            if left_c in need and have[left_c] < need[left_c]:
                formed -= 1
            l += 1

    pass  # return "" if best[0]==inf else s[best[1]:best[2]+1]

In [ ]:
# Uncomment and run when solution is ready
# test_harness(min_window)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Brute force (all substrings) | O(m^2 * n) | O(1) |
| Sliding window no formed | O(m * |need|) | O(|t|) |
| **Sliding window + formed counter** | **O(m + n)** | **O(|t|)** |

Each character is visited at most twice (once by r, once by l).
|t| is at most 52 (upper + lower English letters).

## Real World Connection

Log analysis at Citi requires finding the smallest time window
where a required set of audit events all appear — exactly this
pattern. AWS CloudWatch metric anomaly detection searches for the
shortest interval containing all required signal types. In NLP
pipelines (used in DE feature engineering), finding the shortest
sentence span covering a query's keywords is a direct application.
Network packet inspection tools look for the smallest byte window
containing a full protocol signature — same algorithm.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra